In [1]:
#calculate ha #some ACRES are NULL
import arcpy

fc = r"C:\Users\Melanie\Desktop\Rx_ClassiFIRE\SE_BurnData2010_2023.gdb\ALLStates_BurnData"

# Add field if missing
fields = [f.name for f in arcpy.ListFields(fc)]
if "area_ha" not in fields:
    arcpy.management.AddField(fc, "area_ha", "DOUBLE")

# Calculate hectares
arcpy.management.CalculateField(
    in_table=fc,
    field="area_ha",
    expression="!ACRES! * 0.404685642",
    expression_type="PYTHON3"
)

<Result 'C:\\Users\\Melanie\\Desktop\\Rx_ClassiFIRE\\SE_BurnData2010_2023.gdb\\ALLStates_BurnData'>

<class 'TypeError'>: unsupported operand type(s) for *: 'NoneType' and 'float'

In [2]:
#subset NC
import arcpy

in_fc = r"C:\Users\Melanie\Desktop\Rx_ClassiFIRE\SE_BurnData2010_2023.gdb\ALLStates_BurnData"
out_fc = r"C:\Users\Melanie\Desktop\Rx_ClassiFIRE\SE_BurnData2010_2023.gdb\BurnData_NC"

# Select only NC permits
where = "STATE = 'NC'"

arcpy.analysis.Select(
    in_features=in_fc,
    out_feature_class=out_fc,
    where_clause=where
)

print("NC subset created.")

NC subset created.


In [3]:
# include those only 0.81 ha or larger
import arcpy

in_fc = r"C:\Users\Melanie\Desktop\Rx_ClassiFIRE\SE_BurnData2010_2023.gdb\BurnData_NC"
out_fc = r"C:\Users\Melanie\Desktop\Rx_ClassiFIRE\SE_BurnData2010_2023.gdb\BurnData_NC_081ha"

# Keep only permits ≥ 0.81 ha
where = "area_ha >= 0.81"

arcpy.analysis.Select(
    in_features=in_fc,
    out_feature_class=out_fc,
    where_clause=where
)

print("NC permits ≥ 0.81 ha subset created.")

NC permits ≥ 0.81 ha subset created.


In [4]:
#NC data only available 2014-2023, so limit it to 2014-2022 to match events
import arcpy

in_fc = r"C:\Users\Melanie\Desktop\Rx_ClassiFIRE\SE_BurnData2010_2023.gdb\BurnData_NC_081ha"
out_fc = r"C:\Users\Melanie\Desktop\Rx_ClassiFIRE\SE_BurnData2010_2023.gdb\BurnData_NC_081ha_2014_2022"

# Keep only permits from 2014 through 2022
where = "YEAR >= 2014 AND YEAR <= 2022"

arcpy.analysis.Select(
    in_features=in_fc,
    out_feature_class=out_fc,
    where_clause=where
)

print("NC permits ≥ 0.81 ha, years 2014–2022 subset created.")

NC permits ≥ 0.81 ha, years 2014–2022 subset created.


In [7]:
# I downloaded state boundaries from https://www.census.gov/geographies/mapping-files/time-series/geo/tiger-line-file.html
#Need to subset just NC
import arcpy

states = r"C:\Users\Melanie\Desktop\Rx_ClassiFIRE\state_boundaries\tl_2025_us_state.shp"
nc_boundary = r"C:\Users\Melanie\Desktop\Rx_ClassiFIRE\ClassiFIRE.gdb\NC_boundary"

arcpy.analysis.Select(
    in_features=states,
    out_feature_class=nc_boundary,
    where_clause="STATEFP = '37'"   # 37 = North Carolina FIPS code
)

<Result 'C:\\Users\\Melanie\\Desktop\\Rx_ClassiFIRE\\ClassiFIRE.gdb\\NC_boundary'>

In [1]:
#duplicate events data so I can fill in dates for those that are NULL
import arcpy

gdb = r"C:\Users\Melanie\Desktop\Rx_ClassiFIRE\ClassiFIRE.gdb"

source = f"{gdb}\\SEFM_events_94_22_complete"
target = f"{gdb}\\SEFM_events_94_22_complete_dates_filled"

arcpy.management.CopyFeatures(source, target)

print("Created copy:", target)

Created copy: C:\Users\Melanie\Desktop\Rx_ClassiFIRE\ClassiFIRE.gdb\SEFM_events_94_22_complete_dates_filled


In [2]:
#see if you can go into those zero dates (NULL dates) and replace them with beginning and end of year. 
import arcpy

gdb = r"C:\Users\Melanie\Desktop\Rx_ClassiFIRE\ClassiFIRE.gdb"
events_fc = f"{gdb}\\SEFM_events_94_22_complete_dates_filled"

with arcpy.da.UpdateCursor(
    events_fc,
    ["event_year", "MIN_prebd_min_corrected", "MAX_bd_min_corrected_plus8"]
) as cur:

    for year, tmin, tmax in cur:

        # Skip events with no year (rare)
        if year is None:
            continue

        # Only update events with missing dates
        if tmin is None or tmax is None:

            new_tmin = tmin if tmin is not None else int(f"{year}0101")
            new_tmax = tmax if tmax is not None else int(f"{year}1231")

            cur.updateRow([year, new_tmin, new_tmax])

In [3]:
#Now filter events to only those prescribed
import arcpy

in_fc = r"C:\Users\Melanie\Desktop\Rx_ClassiFIRE\ClassiFIRE.gdb\SEFM_events_94_22_complete_dates_filled"
out_fc = r"C:\Users\Melanie\Desktop\Rx_ClassiFIRE\ClassiFIRE.gdb\SEFM_events_94_22_prescribed"

# Select only prescribed events
where = "buffer_2_5km_final = 'prescribed'"

arcpy.analysis.Select(
    in_features=in_fc,
    out_feature_class=out_fc,
    where_clause=where
)

print("Prescribed-only event subset created.")

Prescribed-only event subset created.


In [4]:
#Clip events to NC
events_in = r"C:\Users\Melanie\Desktop\Rx_ClassiFIRE\ClassiFIRE.gdb\SEFM_events_94_22_prescribed"
events_out = r"C:\Users\Melanie\Desktop\Rx_ClassiFIRE\ClassiFIRE.gdb\SEFM_events_94_22_prescribed_NC"
nc_boundary = r"C:\Users\Melanie\Desktop\Rx_ClassiFIRE\ClassiFIRE.gdb\NC_boundary"

arcpy.analysis.Clip(events_in, nc_boundary, events_out)

<Result 'C:\\Users\\Melanie\\Desktop\\Rx_ClassiFIRE\\ClassiFIRE.gdb\\SEFM_events_94_22_prescribed_NC'>

In [5]:
#include only those events 2014-2022
import arcpy

in_fc = r"C:\Users\Melanie\Desktop\Rx_ClassiFIRE\ClassiFIRE.gdb\SEFM_events_94_22_prescribed_NC"
out_fc = r"C:\Users\Melanie\Desktop\Rx_ClassiFIRE\ClassiFIRE.gdb\SEFM_events_2014_2022_prescribed_NC"

# Keep only events from 2014 through 2022
where = "event_year >= 2014 AND event_year <= 2022"

arcpy.analysis.Select(
    in_features=in_fc,
    out_feature_class=out_fc,
    where_clause=where
)

print("NC prescribed events (2014–2022) subset created.")

NC prescribed events (2014–2022) subset created.


In [6]:
# Create buffers around permit points
import arcpy

permits = r"C:\Users\Melanie\Desktop\Rx_ClassiFIRE\ClassiFIRE_Rx_burn_permits.gdb\BurnData_NC_081ha_2014_2022"
gdb = r"C:\Users\Melanie\Desktop\Rx_ClassiFIRE\ClassiFIRE.gdb"

# Buffer distances in km → ArcGIS distance strings
buffer_distances = {
    "0_5km": "500 Meters",
    "1km": "1000 Meters",
    "1_5km": "1500 Meters",
    "2km": "2000 Meters",
    "2_5km": "2500 Meters",
    "3km": "3000 Meters",
    "3_5km": "3500 Meters",
    "4km": "4000 Meters"
}

for label, dist in buffer_distances.items():
    out_fc = f"{gdb}\\Permits_buf_{label}_NC"
    arcpy.analysis.Buffer(
        in_features=permits,
        out_feature_class=out_fc,
        buffer_distance_or_field=dist,
        dissolve_option="NONE"
    )
    print(f"Created buffer: {label}")

Created buffer: 0_5km
Created buffer: 1km
Created buffer: 1_5km
Created buffer: 2km
Created buffer: 2_5km
Created buffer: 3km
Created buffer: 3_5km
Created buffer: 4km


In [7]:
#add fields to permit layer
import arcpy

permits = r"C:\Users\Melanie\Desktop\Rx_ClassiFIRE\ClassiFIRE_Rx_burn_permits.gdb\BurnData_NC_081ha_2014_2022"

buffer_labels = [
    "0_5km",
    "1km",
    "1_5km",
    "2km",
    "2_5km",
    "3km",
    "3_5km",  
    "4km"     
]


for label in buffer_labels:
    field = f"buffer_{label}"
    if field not in [f.name for f in arcpy.ListFields(permits)]:
        arcpy.management.AddField(permits, field, "TEXT")
        print(f"Added field {field}")

In [8]:
#spatial join
import arcpy

gdb = r"C:\Users\Melanie\Desktop\Rx_ClassiFIRE\ClassiFIRE.gdb"
events = f"{gdb}\\SEFM_events_2014_2022_prescribed_NC"

buffer_labels = [
    "0_5km",
    "1km",
    "1_5km",
    "2km",
    "2_5km",
    "3km",
    "3_5km",
    "4km"
]

for label in buffer_labels:
    permits_buf = f"{gdb}\\Permits_buf_{label}_NC"
    sj = f"{gdb}\\PermitEventJoin_{label}_NC"

    arcpy.analysis.SpatialJoin(
        target_features=permits_buf,
        join_features=events,
        out_feature_class=sj,
        join_operation="JOIN_ONE_TO_MANY",
        match_option="INTERSECT"
    )

    print(f"Spatial join completed for buffer {label}")

Spatial join completed for buffer 0_5km
Spatial join completed for buffer 1km
Spatial join completed for buffer 1_5km
Spatial join completed for buffer 2km
Spatial join completed for buffer 2_5km
Spatial join completed for buffer 3km
Spatial join completed for buffer 3_5km
Spatial join completed for buffer 4km


In [9]:
#Temporal checking
import arcpy
from datetime import datetime

# Paths
gdb = r"C:\Users\Melanie\Desktop\Rx_ClassiFIRE\ClassiFIRE.gdb"
permits_fc = r"C:\Users\Melanie\Desktop\Rx_ClassiFIRE\ClassiFIRE_Rx_burn_permits.gdb\BurnData_NC_081ha_2014_2022"

# Buffer labels
buffer_labels = [
    "0_5km",
    "1km",
    "1_5km",
    "2km",
    "2_5km",
    "3km",
    "3_5km",
    "4km"
]

# --- DATE PARSERS ---

# SEFM event dates: YYYYMMDD integer
def parse_sefm_date(val):
    s = str(val)
    year = int(s[0:4])
    month = int(s[4:6])
    day = int(s[6:8])
    return datetime(year, month, day)

# Permit dates: MM/DD/YYYY (may have single-digit month/day OR already datetime)
def parse_permit_date(val):
    if isinstance(val, datetime):
        return val
    return datetime.strptime(val, "%m/%d/%Y")

# --- MAIN LOOP ---

for label in buffer_labels:

    print(f"Processing temporal matching for buffer {label}...")

    # Spatial join result for this buffer
    sj = f"{gdb}\\PermitEventJoin_{label}_NC"

    # Build mapping: permit OBJECTID → list of (event_start, event_end)
    permit_to_events = {}

    with arcpy.da.SearchCursor(
        sj,
        ["TARGET_FID", "MIN_prebd_min_corrected", "MAX_bd_min_corrected_plus8"]
    ) as cur:
        for pid, tmin_raw, tmax_raw in cur:
            if tmin_raw is None or tmax_raw is None:
                continue
            tmin = parse_sefm_date(tmin_raw)
            tmax = parse_sefm_date(tmax_raw)
            permit_to_events.setdefault(pid, []).append((tmin, tmax))

    # Update the permit layer
    field = f"buffer_{label}"

    with arcpy.da.UpdateCursor(
        permits_fc,
        ["OBJECTID", "DATE", field]
    ) as cur:

        for pid, permit_date_raw, _ in cur:

            permit_date = parse_permit_date(permit_date_raw)

            # Default: no spatial match → no_match
            classification = "no_match"

            # Only check temporal overlap if spatial matches exist
            if pid in permit_to_events:
                for (tmin, tmax) in permit_to_events[pid]:
                    if tmin <= permit_date <= tmax:
                        classification = "match"
                        break

            cur.updateRow([pid, permit_date_raw, classification])

    print(f"Completed buffer {label}")

print("All temporal matching complete.")

Processing temporal matching for buffer 0_5km...
Completed buffer 0_5km
Processing temporal matching for buffer 1km...
Completed buffer 1km
Processing temporal matching for buffer 1_5km...
Completed buffer 1_5km
Processing temporal matching for buffer 2km...
Completed buffer 2km
Processing temporal matching for buffer 2_5km...
Completed buffer 2_5km
Processing temporal matching for buffer 3km...
Completed buffer 3km
Processing temporal matching for buffer 3_5km...
Completed buffer 3_5km
Processing temporal matching for buffer 4km...
Completed buffer 4km
All temporal matching complete.


In [10]:
#Count percentage of permits that match
import arcpy

permits_fc = r"C:\Users\Melanie\Desktop\Rx_ClassiFIRE\ClassiFIRE_Rx_burn_permits.gdb\BurnData_NC_081ha_2014_2022"

buffer_labels = [
    "0_5km",
    "1km",
    "1_5km",
    "2km",
    "2_5km",
    "3km",
    "3_5km",
    "4km"
]

# Count total permits
total = int(arcpy.management.GetCount(permits_fc)[0])

results = {}

for label in buffer_labels:
    field = f"buffer_{label}"
    match_count = 0

    with arcpy.da.SearchCursor(permits_fc, [field]) as cur:
        for (val,) in cur:
            if val == "match":
                match_count += 1

    pct = (match_count / total) * 100
    results[label] = pct
    print(f"{label}: {pct:.2f}% match")

print("\nSummary:", results)

0_5km: 35.62% match
1km: 42.58% match
1_5km: 46.68% match
2km: 49.80% match
2_5km: 52.36% match
3km: 54.83% match
3_5km: 57.32% match
4km: 59.70% match

Summary: {'0_5km': 35.61549891416162, '1km': 42.57629443364956, '1_5km': 46.67962052806035, '2km': 49.7999771402446, '2_5km': 52.36026974511373, '3km': 54.82912332838039, '3_5km': 57.32083666704766, '4km': 59.698251228711854}


In [11]:
#Now check NC events 
import arcpy

events_fc = r"C:\Users\Melanie\Desktop\Rx_ClassiFIRE\ClassiFIRE.gdb\SEFM_events_2014_2022_prescribed_NC"

buffer_labels = [
    "0_5km", "1km", "1_5km", "2km",
    "2_5km", "3km", "3_5km", "4km"
]

for label in buffer_labels:
    field = f"evt_buf_{label}"
    if field not in [f.name for f in arcpy.ListFields(events_fc)]:
        arcpy.management.AddField(events_fc, field, "TEXT")
        print(f"Added field {field}")

Added field evt_buf_0_5km
Added field evt_buf_1km
Added field evt_buf_1_5km
Added field evt_buf_2km
Added field evt_buf_2_5km
Added field evt_buf_3km
Added field evt_buf_3_5km
Added field evt_buf_4km


In [12]:
import arcpy
from datetime import datetime

# Paths
gdb = r"C:\Users\Melanie\Desktop\Rx_ClassiFIRE\ClassiFIRE.gdb"
events_fc = f"{gdb}\\SEFM_events_2014_2022_prescribed_NC"

# Buffer labels
buffer_labels = [
    "0_5km", "1km", "1_5km", "2km",
    "2_5km", "3km", "3_5km", "4km"
]

# --- DATE PARSERS ---

# Event dates: YYYYMMDD integer
def parse_sefm_date(val):
    s = str(val)
    return datetime(int(s[0:4]), int(s[4:6]), int(s[6:8]))

# Permit dates: MM/DD/YYYY or datetime
def parse_permit_date(val):
    if isinstance(val, datetime):
        return val
    return datetime.strptime(val, "%m/%d/%Y")

# --- MAIN LOOP ---

for label in buffer_labels:

    print(f"Processing event matching for buffer {label}...")

    # Spatial join result for this buffer
    sj = f"{gdb}\\PermitEventJoin_{label}_NC"

    # Build mapping: event_id → list of permit dates
    event_to_permits = {}

    with arcpy.da.SearchCursor(
        sj,
        ["JOIN_FID", "DATE"]   # JOIN_FID = event OBJECTID
    ) as cur:
        for eid, permit_date_raw in cur:
            if permit_date_raw is None:
                continue
            permit_date = parse_permit_date(permit_date_raw)
            event_to_permits.setdefault(eid, []).append(permit_date)

    # Update event layer
    field = f"evt_buf_{label}"

    with arcpy.da.UpdateCursor(
        events_fc,
        ["OBJECTID", "MIN_prebd_min_corrected", "MAX_bd_min_corrected_plus8", field]
    ) as cur:

        for eid, tmin_raw, tmax_raw, _ in cur:

            # If event has no valid temporal window → cannot match
            if tmin_raw is None or tmax_raw is None:
                cur.updateRow([eid, tmin_raw, tmax_raw, "no_match"])
                continue

            # Parse event dates
            tmin = parse_sefm_date(tmin_raw)
            tmax = parse_sefm_date(tmax_raw)

            classification = "no_match"

            # Only check temporal overlap if spatial matches exist
            if eid in event_to_permits:
                for permit_date in event_to_permits[eid]:
                    if tmin <= permit_date <= tmax:
                        classification = "match"
                        break

            cur.updateRow([eid, tmin_raw, tmax_raw, classification])

    print(f"Completed buffer {label}")

print("All event matching complete.")

Processing event matching for buffer 0_5km...
Completed buffer 0_5km
Processing event matching for buffer 1km...
Completed buffer 1km
Processing event matching for buffer 1_5km...
Completed buffer 1_5km
Processing event matching for buffer 2km...
Completed buffer 2km
Processing event matching for buffer 2_5km...
Completed buffer 2_5km
Processing event matching for buffer 3km...
Completed buffer 3km
Processing event matching for buffer 3_5km...
Completed buffer 3_5km
Processing event matching for buffer 4km...
Completed buffer 4km
All event matching complete.


In [13]:
#percentage of events matched at each buffer size
import arcpy

events_fc = r"C:\Users\Melanie\Desktop\Rx_ClassiFIRE\ClassiFIRE.gdb\SEFM_events_2014_2022_prescribed_NC"

buffer_labels = [
    "0_5km",
    "1km",
    "1_5km",
    "2km",
    "2_5km",
    "3km",
    "3_5km",
    "4km"
]

# Count total events
total = int(arcpy.management.GetCount(events_fc)[0])

results = {}

for label in buffer_labels:
    field = f"evt_buf_{label}"
    match_count = 0

    with arcpy.da.SearchCursor(events_fc, [field]) as cur:
        for (val,) in cur:
            if val == "match":
                match_count += 1

    pct = (match_count / total) * 100
    results[label] = pct
    print(f"{label}: {pct:.2f}% matched")

print("\nSummary:", results)

0_5km: 5.88% matched
1km: 7.96% matched
1_5km: 9.37% matched
2km: 10.36% matched
2_5km: 11.29% matched
3km: 12.22% matched
3_5km: 13.19% matched
4km: 14.16% matched

Summary: {'0_5km': 5.883106621965021, '1km': 7.9587435136347136, '1_5km': 9.36599115932435, '2km': 10.356830169339512, '2_5km': 11.285741741228726, '3km': 12.223195028721518, '3_5km': 13.186273463025048, '4km': 14.157893612932156}


In [1]:
#Now create subset of permits for AL, FL, MS
import arcpy

in_fc = r"C:\Users\Melanie\Desktop\Rx_ClassiFIRE\SE_BurnData2010_2023.gdb\ALLStates_BurnData"
gdb = r"C:\Users\Melanie\Desktop\Rx_ClassiFIRE\SE_BurnData2010_2023.gdb"

states = ["MS", "AL", "FL"]

for st in states:
    out_fc = fr"{gdb}\BurnData_{st}"
    where = f"STATE = '{st}'"

    arcpy.analysis.Select(
        in_features=in_fc,
        out_feature_class=out_fc,
        where_clause=where
    )

    print(f"{st} subset created.")

MS subset created.
AL subset created.
FL subset created.


In [2]:
#Alabama-create subsets of data 1) all permits 2011-2022, and 2) permits where area is greater than or equal to 0.81 ha
#AL data only available 2011-2023, so limit it to 2011-2022 to match events

import arcpy

gdb = r"C:\Users\Melanie\Desktop\Rx_ClassiFIRE\SE_BurnData2010_2023.gdb"
al_fc = fr"{gdb}\BurnData_AL"

al_11_22_all = fr"{gdb}\AL_permits_2011_2022_all"

where_yrs = "YEAR >= 2011 AND YEAR <= 2022"

arcpy.analysis.Select(
    in_features=al_fc,
    out_feature_class=al_11_22_all,
    where_clause=where_yrs
)

print("Created:", al_11_22_all)


al_11_22_big = fr"{gdb}\AL_permits_2011_2022_ge0_81ha"

where_big = "YEAR >= 2011 AND YEAR <= 2022 AND area_ha >= 0.81"

arcpy.analysis.Select(
    in_features=al_fc,
    out_feature_class=al_11_22_big,
    where_clause=where_big
)

print("Created:", al_11_22_big)


Created: C:\Users\Melanie\Desktop\Rx_ClassiFIRE\SE_BurnData2010_2023.gdb\AL_permits_2011_2022_all
Created: C:\Users\Melanie\Desktop\Rx_ClassiFIRE\SE_BurnData2010_2023.gdb\AL_permits_2011_2022_ge0_81ha


In [3]:
#Create buffers around points for each ALabama permit subset: 
import arcpy

# Input datasets (still in SE_BurnData2010_2023.gdb)
gdb_in = r"C:\Users\Melanie\Desktop\Rx_ClassiFIRE\SE_BurnData2010_2023.gdb"
al_all = fr"{gdb_in}\AL_permits_2011_2022_all"
al_big = fr"{gdb_in}\AL_permits_2011_2022_ge0_81ha"

# Output geodatabase (same as NC permit buffers)
gdb_out = r"C:\Users\Melanie\Desktop\Rx_ClassiFIRE\ClassiFIRE_Rx_burn_permits.gdb"

datasets = {
    "AL_all": al_all,
    "AL_big": al_big
}

buffer_distances = {
    "0_5km": "500 Meters",
    "1km": "1000 Meters",
    "1_5km": "1500 Meters",
    "2km": "2000 Meters",
    "2_5km": "2500 Meters",
    "3km": "3000 Meters",
    "3_5km": "3500 Meters",
    "4km": "4000 Meters"
}

for label, permits in datasets.items():
    for dist_label, dist in buffer_distances.items():

        out_fc = fr"{gdb_out}\Permits_buf_{dist_label}_{label}"

        arcpy.analysis.Buffer(
            in_features=permits,
            out_feature_class=out_fc,
            buffer_distance_or_field=dist,
            dissolve_option="NONE"
        )

        print(f"Created buffer: {out_fc}")

Created buffer: C:\Users\Melanie\Desktop\Rx_ClassiFIRE\ClassiFIRE_Rx_burn_permits.gdb\Permits_buf_0_5km_AL_all
Created buffer: C:\Users\Melanie\Desktop\Rx_ClassiFIRE\ClassiFIRE_Rx_burn_permits.gdb\Permits_buf_1km_AL_all
Created buffer: C:\Users\Melanie\Desktop\Rx_ClassiFIRE\ClassiFIRE_Rx_burn_permits.gdb\Permits_buf_1_5km_AL_all
Created buffer: C:\Users\Melanie\Desktop\Rx_ClassiFIRE\ClassiFIRE_Rx_burn_permits.gdb\Permits_buf_2km_AL_all
Created buffer: C:\Users\Melanie\Desktop\Rx_ClassiFIRE\ClassiFIRE_Rx_burn_permits.gdb\Permits_buf_2_5km_AL_all
Created buffer: C:\Users\Melanie\Desktop\Rx_ClassiFIRE\ClassiFIRE_Rx_burn_permits.gdb\Permits_buf_3km_AL_all
Created buffer: C:\Users\Melanie\Desktop\Rx_ClassiFIRE\ClassiFIRE_Rx_burn_permits.gdb\Permits_buf_3_5km_AL_all
Created buffer: C:\Users\Melanie\Desktop\Rx_ClassiFIRE\ClassiFIRE_Rx_burn_permits.gdb\Permits_buf_4km_AL_all
Created buffer: C:\Users\Melanie\Desktop\Rx_ClassiFIRE\ClassiFIRE_Rx_burn_permits.gdb\Permits_buf_0_5km_AL_big
Created b

In [4]:
#get boundary for AL

import arcpy

states = r"C:\Users\Melanie\Desktop\Rx_ClassiFIRE\state_boundaries\tl_2025_us_state.shp"

# Output boundary goes to the same gdb as your permit buffers
al_boundary = r"C:\Users\Melanie\Desktop\Rx_ClassiFIRE\ClassiFIRE_Rx_burn_permits.gdb\AL_boundary"

arcpy.analysis.Select(
    in_features=states,
    out_feature_class=al_boundary,
    where_clause="STATEFP = '01'"   # 01 = Alabama FIPS code
)

print("Created Alabama boundary:", al_boundary)

Created Alabama boundary: C:\Users\Melanie\Desktop\Rx_ClassiFIRE\ClassiFIRE_Rx_burn_permits.gdb\AL_boundary


In [5]:
#Clip Rx events to AL:
import arcpy

# Input prescribed-fire events (full Rx dataset)
events_in = r"C:\Users\Melanie\Desktop\Rx_ClassiFIRE\ClassiFIRE.gdb\SEFM_events_94_22_prescribed"

# Output: Alabama-only prescribed events, stored in the same gdb as permit buffers
events_out = r"C:\Users\Melanie\Desktop\Rx_ClassiFIRE\ClassiFIRE_Rx_burn_permits.gdb\SEFM_events_94_22_prescribed_AL"

# Alabama boundary (already created)
al_boundary = r"C:\Users\Melanie\Desktop\Rx_ClassiFIRE\ClassiFIRE_Rx_burn_permits.gdb\AL_boundary"

arcpy.analysis.Clip(
    in_features=events_in,
    clip_features=al_boundary,
    out_feature_class=events_out
)

print("Created Alabama prescribed events subset:", events_out)

Created Alabama prescribed events subset: C:\Users\Melanie\Desktop\Rx_ClassiFIRE\ClassiFIRE_Rx_burn_permits.gdb\SEFM_events_94_22_prescribed_AL


In [6]:
#include only those Alabama events 2011-2022
import arcpy

# Input: AL prescribed events (already clipped to AL)
in_fc = r"C:\Users\Melanie\Desktop\Rx_ClassiFIRE\ClassiFIRE_Rx_burn_permits.gdb\SEFM_events_94_22_prescribed_AL"

# Output: AL prescribed events limited to 2011–2022
out_fc = r"C:\Users\Melanie\Desktop\Rx_ClassiFIRE\ClassiFIRE_Rx_burn_permits.gdb\SEFM_events_2011_2022_prescribed_AL"

# Keep only events from 2011 through 2022
where = "event_year >= 2011 AND event_year <= 2022"

arcpy.analysis.Select(
    in_features=in_fc,
    out_feature_class=out_fc,
    where_clause=where
)

print("AL prescribed events (2011–2022) subset created.")

AL prescribed events (2011–2022) subset created.


In [8]:
#copy AL permits to ClassiFIRE_Rx_burn_permits.gdb 
import arcpy

gdb_in = r"C:\Users\Melanie\Desktop\Rx_ClassiFIRE\SE_BurnData2010_2023.gdb"
gdb_out = r"C:\Users\Melanie\Desktop\Rx_ClassiFIRE\ClassiFIRE_Rx_burn_permits.gdb"

inputs = [
    "AL_permits_2011_2022_all",
    "AL_permits_2011_2022_ge0_81ha"
]

for name in inputs:
    src = fr"{gdb_in}\{name}"
    dst = fr"{gdb_out}\{name}"
    arcpy.management.CopyFeatures(src, dst)
    print("Copied:", dst)

Copied: C:\Users\Melanie\Desktop\Rx_ClassiFIRE\ClassiFIRE_Rx_burn_permits.gdb\AL_permits_2011_2022_all
Copied: C:\Users\Melanie\Desktop\Rx_ClassiFIRE\ClassiFIRE_Rx_burn_permits.gdb\AL_permits_2011_2022_ge0_81ha


In [9]:
#add buffer fields to both Alabama permit datasets
import arcpy

gdb = r"C:\Users\Melanie\Desktop\Rx_ClassiFIRE\ClassiFIRE_Rx_burn_permits.gdb"

# Alabama permit datasets
permit_layers = [
    fr"{gdb}\AL_permits_2011_2022_all",
    fr"{gdb}\AL_permits_2011_2022_ge0_81ha"
]

# Buffer labels (same as NC)
buffer_labels = [
    "0_5km",
    "1km",
    "1_5km",
    "2km",
    "2_5km",
    "3km",
    "3_5km",
    "4km"
]

# Add fields to each permit layer
for permits in permit_layers:
    existing_fields = [f.name for f in arcpy.ListFields(permits)]
    
    for label in buffer_labels:
        field = f"buffer_{label}"
        
        if field not in existing_fields:
            arcpy.management.AddField(permits, field, "TEXT")
            print(f"Added field {field} to {permits}")
        else:
            print(f"Field {field} already exists in {permits}")

Added field buffer_0_5km to C:\Users\Melanie\Desktop\Rx_ClassiFIRE\ClassiFIRE_Rx_burn_permits.gdb\AL_permits_2011_2022_all
Added field buffer_1km to C:\Users\Melanie\Desktop\Rx_ClassiFIRE\ClassiFIRE_Rx_burn_permits.gdb\AL_permits_2011_2022_all
Added field buffer_1_5km to C:\Users\Melanie\Desktop\Rx_ClassiFIRE\ClassiFIRE_Rx_burn_permits.gdb\AL_permits_2011_2022_all
Added field buffer_2km to C:\Users\Melanie\Desktop\Rx_ClassiFIRE\ClassiFIRE_Rx_burn_permits.gdb\AL_permits_2011_2022_all
Added field buffer_2_5km to C:\Users\Melanie\Desktop\Rx_ClassiFIRE\ClassiFIRE_Rx_burn_permits.gdb\AL_permits_2011_2022_all
Added field buffer_3km to C:\Users\Melanie\Desktop\Rx_ClassiFIRE\ClassiFIRE_Rx_burn_permits.gdb\AL_permits_2011_2022_all
Added field buffer_3_5km to C:\Users\Melanie\Desktop\Rx_ClassiFIRE\ClassiFIRE_Rx_burn_permits.gdb\AL_permits_2011_2022_all
Added field buffer_4km to C:\Users\Melanie\Desktop\Rx_ClassiFIRE\ClassiFIRE_Rx_burn_permits.gdb\AL_permits_2011_2022_all
Added field buffer_0_5km

In [10]:
#spatial join for each dataset 
import arcpy

gdb = r"C:\Users\Melanie\Desktop\Rx_ClassiFIRE\ClassiFIRE_Rx_burn_permits.gdb"

# Alabama prescribed events (clipped + year-filtered)
events = fr"{gdb}\SEFM_events_2011_2022_prescribed_AL"

# Two Alabama permit datasets
permit_sets = [
    "AL_all",
    "AL_big"
]

# Buffer labels (same as NC)
buffer_labels = [
    "0_5km",
    "1km",
    "1_5km",
    "2km",
    "2_5km",
    "3km",
    "3_5km",
    "4km"
]

# Spatial join for each permit set × each buffer distance
for pset in permit_sets:
    for label in buffer_labels:

        permits_buf = fr"{gdb}\Permits_buf_{label}_{pset}"
        sj = fr"{gdb}\PermitEventJoin_{label}_{pset}"

        arcpy.analysis.SpatialJoin(
            target_features=permits_buf,
            join_features=events,
            out_feature_class=sj,
            join_operation="JOIN_ONE_TO_MANY",
            match_option="INTERSECT"
        )

        print(f"Spatial join completed: {sj}")

Spatial join completed: C:\Users\Melanie\Desktop\Rx_ClassiFIRE\ClassiFIRE_Rx_burn_permits.gdb\PermitEventJoin_0_5km_AL_all
Spatial join completed: C:\Users\Melanie\Desktop\Rx_ClassiFIRE\ClassiFIRE_Rx_burn_permits.gdb\PermitEventJoin_1km_AL_all
Spatial join completed: C:\Users\Melanie\Desktop\Rx_ClassiFIRE\ClassiFIRE_Rx_burn_permits.gdb\PermitEventJoin_1_5km_AL_all
Spatial join completed: C:\Users\Melanie\Desktop\Rx_ClassiFIRE\ClassiFIRE_Rx_burn_permits.gdb\PermitEventJoin_2km_AL_all
Spatial join completed: C:\Users\Melanie\Desktop\Rx_ClassiFIRE\ClassiFIRE_Rx_burn_permits.gdb\PermitEventJoin_2_5km_AL_all
Spatial join completed: C:\Users\Melanie\Desktop\Rx_ClassiFIRE\ClassiFIRE_Rx_burn_permits.gdb\PermitEventJoin_3km_AL_all
Spatial join completed: C:\Users\Melanie\Desktop\Rx_ClassiFIRE\ClassiFIRE_Rx_burn_permits.gdb\PermitEventJoin_3_5km_AL_all
Spatial join completed: C:\Users\Melanie\Desktop\Rx_ClassiFIRE\ClassiFIRE_Rx_burn_permits.gdb\PermitEventJoin_4km_AL_all
Spatial join completed: 

In [11]:
#temporal matching for both AL datasets
import arcpy
from datetime import datetime

# Paths
gdb = r"C:\Users\Melanie\Desktop\Rx_ClassiFIRE\ClassiFIRE_Rx_burn_permits.gdb"

# Alabama permit datasets
permit_sets = {
    "AL_all": fr"{gdb}\AL_permits_2011_2022_all",
    "AL_big": fr"{gdb}\AL_permits_2011_2022_ge0_81ha"
}

# Buffer labels
buffer_labels = [
    "0_5km",
    "1km",
    "1_5km",
    "2km",
    "2_5km",
    "3km",
    "3_5km",
    "4km"
]

# --- DATE PARSERS ---

# SEFM event dates: YYYYMMDD integer
def parse_sefm_date(val):
    s = str(val)
    year = int(s[0:4])
    month = int(s[4:6])
    day = int(s[6:8])
    return datetime(year, month, day)

# Permit dates: MM/DD/YYYY or datetime
def parse_permit_date(val):
    if isinstance(val, datetime):
        return val
    return datetime.strptime(val, "%m/%d/%Y")

# --- MAIN LOOP FOR BOTH PERMIT DATASETS ---

for pset_label, permits_fc in permit_sets.items():

    print(f"\n=== Processing permit set: {pset_label} ===")

    for label in buffer_labels:

        print(f"  Buffer {label}...")

        # Spatial join table for this permit set + buffer
        sj = fr"{gdb}\PermitEventJoin_{label}_{pset_label}"

        # Build mapping: permit OBJECTID → list of (event_start, event_end)
        permit_to_events = {}

        with arcpy.da.SearchCursor(
            sj,
            ["TARGET_FID", "MIN_prebd_min_corrected", "MAX_bd_min_corrected_plus8"]
        ) as cur:
            for pid, tmin_raw, tmax_raw in cur:
                if tmin_raw is None or tmax_raw is None:
                    continue
                tmin = parse_sefm_date(tmin_raw)
                tmax = parse_sefm_date(tmax_raw)
                permit_to_events.setdefault(pid, []).append((tmin, tmax))

        # Field to update on the permit layer
        field = f"buffer_{label}"

        with arcpy.da.UpdateCursor(
            permits_fc,
            ["OBJECTID", "DATE", field]
        ) as cur:

            for pid, permit_date_raw, _ in cur:

                permit_date = parse_permit_date(permit_date_raw)

                # Default: no spatial match → no_match
                classification = "no_match"

                # Only check temporal overlap if spatial matches exist
                if pid in permit_to_events:
                    for (tmin, tmax) in permit_to_events[pid]:
                        if tmin <= permit_date <= tmax:
                            classification = "match"
                            break

                cur.updateRow([pid, permit_date_raw, classification])

        print(f"    Completed buffer {label}")

print("\nAll temporal matching complete for Alabama.")


=== Processing permit set: AL_all ===
  Buffer 0_5km...
    Completed buffer 0_5km
  Buffer 1km...
    Completed buffer 1km
  Buffer 1_5km...
    Completed buffer 1_5km
  Buffer 2km...
    Completed buffer 2km
  Buffer 2_5km...
    Completed buffer 2_5km
  Buffer 3km...
    Completed buffer 3km
  Buffer 3_5km...
    Completed buffer 3_5km
  Buffer 4km...
    Completed buffer 4km

=== Processing permit set: AL_big ===
  Buffer 0_5km...
    Completed buffer 0_5km
  Buffer 1km...
    Completed buffer 1km
  Buffer 1_5km...
    Completed buffer 1_5km
  Buffer 2km...
    Completed buffer 2km
  Buffer 2_5km...
    Completed buffer 2_5km
  Buffer 3km...
    Completed buffer 3km
  Buffer 3_5km...
    Completed buffer 3_5km
  Buffer 4km...
    Completed buffer 4km

All temporal matching complete for Alabama.


In [12]:
#Calculate % matching for each dataset
import arcpy

gdb = r"C:\Users\Melanie\Desktop\Rx_ClassiFIRE\ClassiFIRE_Rx_burn_permits.gdb"

# Two Alabama permit datasets
permit_sets = {
    "AL_all": fr"{gdb}\AL_permits_2011_2022_all",
    "AL_big": fr"{gdb}\AL_permits_2011_2022_ge0_81ha"
}

buffer_labels = [
    "0_5km",
    "1km",
    "1_5km",
    "2km",
    "2_5km",
    "3km",
    "3_5km",
    "4km"
]

for label, permits_fc in permit_sets.items():

    print(f"\n=== Match percentages for {label} ===")

    total = int(arcpy.management.GetCount(permits_fc)[0])
    results = {}

    for buf in buffer_labels:
        field = f"buffer_{buf}"
        match_count = 0

        with arcpy.da.SearchCursor(permits_fc, [field]) as cur:
            for (val,) in cur:
                if val == "match":
                    match_count += 1

        pct = (match_count / total) * 100
        results[buf] = pct
        print(f"{buf}: {pct:.2f}% match")

    print("Summary:", results)


=== Match percentages for AL_all ===
0_5km: 15.20% match
1km: 22.91% match
1_5km: 27.97% match
2km: 32.47% match
2_5km: 36.80% match
3km: 41.12% match
3_5km: 45.26% match
4km: 48.92% match
Summary: {'0_5km': 15.197773521358512, '1km': 22.913319021480635, '1_5km': 27.97367304701058, '2km': 32.47097993341132, '2_5km': 36.797315884871004, '3km': 41.121723592703525, '3_5km': 45.2648763995835, '4km': 48.920183568793306}

=== Match percentages for AL_big ===
0_5km: 18.22% match
1km: 27.20% match
1_5km: 32.84% match
2km: 37.73% match
2_5km: 42.31% match
3km: 46.91% match
3_5km: 51.26% match
4km: 55.05% match
Summary: {'0_5km': 18.218110236220472, '1km': 27.195275590551184, '1_5km': 32.83937007874016, '2km': 37.73464566929134, '2_5km': 42.309448818897636, '3km': 46.91338582677165, '3_5km': 51.25905511811024, '4km': 55.0511811023622}


In [13]:
#Calculate % of AL Rx events that match AL_big permits at each buffer size
import arcpy

gdb = r"C:\Users\Melanie\Desktop\Rx_ClassiFIRE\ClassiFIRE_Rx_burn_permits.gdb"
events_fc = fr"{gdb}\SEFM_events_2011_2022_prescribed_AL"

buffer_labels = [
    "0_5km", "1km", "1_5km", "2km",
    "2_5km", "3km", "3_5km", "4km"
]

existing = [f.name for f in arcpy.ListFields(events_fc)]

for label in buffer_labels:
    field = f"evt_buf_{label}"
    if field not in existing:
        arcpy.management.AddField(events_fc, field, "TEXT")
        print(f"Added field {field}")

Added field evt_buf_0_5km
Added field evt_buf_1km
Added field evt_buf_1_5km
Added field evt_buf_2km
Added field evt_buf_2_5km
Added field evt_buf_3km
Added field evt_buf_3_5km
Added field evt_buf_4km


In [14]:
import arcpy
from datetime import datetime

gdb = r"C:\Users\Melanie\Desktop\Rx_ClassiFIRE\ClassiFIRE_Rx_burn_permits.gdb"
events_fc = fr"{gdb}\SEFM_events_2011_2022_prescribed_AL"

buffer_labels = [
    "0_5km", "1km", "1_5km", "2km",
    "2_5km", "3km", "3_5km", "4km"
]

def parse_sefm_date(val):
    s = str(val)
    return datetime(int(s[0:4]), int(s[4:6]), int(s[6:8]))

def parse_permit_date(val):
    if isinstance(val, datetime):
        return val
    return datetime.strptime(val, "%m/%d/%Y")

for label in buffer_labels:

    print(f"Processing event matching for buffer {label}...")

    sj = fr"{gdb}\PermitEventJoin_{label}_AL_big"

    event_to_permits = {}

    with arcpy.da.SearchCursor(
        sj,
        ["JOIN_FID", "DATE"]
    ) as cur:
        for eid, permit_date_raw in cur:
            if permit_date_raw is None:
                continue
            permit_date = parse_permit_date(permit_date_raw)
            event_to_permits.setdefault(eid, []).append(permit_date)

    field = f"evt_buf_{label}"

    with arcpy.da.UpdateCursor(
        events_fc,
        ["OBJECTID", "MIN_prebd_min_corrected", "MAX_bd_min_corrected_plus8", field]
    ) as cur:

        for eid, tmin_raw, tmax_raw, _ in cur:

            if tmin_raw is None or tmax_raw is None:
                cur.updateRow([eid, tmin_raw, tmax_raw, "no_match"])
                continue

            tmin = parse_sefm_date(tmin_raw)
            tmax = parse_sefm_date(tmax_raw)

            classification = "no_match"

            if eid in event_to_permits:
                for permit_date in event_to_permits[eid]:
                    if tmin <= permit_date <= tmax:
                        classification = "match"
                        break

            cur.updateRow([eid, tmin_raw, tmax_raw, classification])

    print(f"Completed buffer {label}")

print("All Alabama event matching complete.")

Processing event matching for buffer 0_5km...
Completed buffer 0_5km
Processing event matching for buffer 1km...
Completed buffer 1km
Processing event matching for buffer 1_5km...
Completed buffer 1_5km
Processing event matching for buffer 2km...
Completed buffer 2km
Processing event matching for buffer 2_5km...
Completed buffer 2_5km
Processing event matching for buffer 3km...
Completed buffer 3km
Processing event matching for buffer 3_5km...
Completed buffer 3_5km
Processing event matching for buffer 4km...
Completed buffer 4km
All Alabama event matching complete.


In [15]:
import arcpy

gdb = r"C:\Users\Melanie\Desktop\Rx_ClassiFIRE\ClassiFIRE_Rx_burn_permits.gdb"
events_fc = fr"{gdb}\SEFM_events_2011_2022_prescribed_AL"

buffer_labels = [
    "0_5km", "1km", "1_5km", "2km",
    "2_5km", "3km", "3_5km", "4km"
]

total = int(arcpy.management.GetCount(events_fc)[0])
results = {}

for label in buffer_labels:
    field = f"evt_buf_{label}"
    match_count = 0

    with arcpy.da.SearchCursor(events_fc, [field]) as cur:
        for (val,) in cur:
            if val == "match":
                match_count += 1

    pct = (match_count / total) * 100
    results[label] = pct
    print(f"{label}: {pct:.2f}% matched")

print("\nSummary:", results)

0_5km: 23.16% matched
1km: 37.45% matched
1_5km: 44.96% matched
2km: 50.15% matched
2_5km: 54.28% matched
3km: 57.92% matched
3_5km: 61.09% matched
4km: 63.95% matched

Summary: {'0_5km': 23.16149191652123, '1km': 37.447355495406995, '1_5km': 44.95501050277461, '2km': 50.15310014735236, '2_5km': 54.28210139096448, '3km': 57.916792943807536, '3_5km': 61.09375163289407, '4km': 63.949879296470854}


In [16]:
#Now Mississippi. First, copy permit data over to permit gdb:
import arcpy

gdb_in  = r"C:\Users\Melanie\Desktop\Rx_ClassiFIRE\SE_BurnData2010_2023.gdb"
gdb_out = r"C:\Users\Melanie\Desktop\Rx_ClassiFIRE\ClassiFIRE_Rx_burn_permits.gdb"

src = fr"{gdb_in}\BurnData_MS"
dst = fr"{gdb_out}\BurnData_MS"

arcpy.management.CopyFeatures(src, dst)

print("Copied BurnData_MS into permit geodatabase.")

Copied BurnData_MS into permit geodatabase.


In [28]:
#Only use MS permits 2010-2022 to match events (exclude those where DATE == NULL; 5% of records)
import arcpy

gdb = r"C:\Users\Melanie\Desktop\Rx_ClassiFIRE\ClassiFIRE_Rx_burn_permits.gdb"

ms_permits = fr"{gdb}\BurnData_MS"
ms_2010_2022 = fr"{gdb}\MS_permits_2010_2022"

# Keep only permits from 2010–2022 AND with a valid DATE
where = "YEAR >= 2010 AND YEAR <= 2022 AND DATE IS NOT NULL"

arcpy.analysis.Select(
    in_features=ms_permits,
    out_feature_class=ms_2010_2022,
    where_clause=where
)

print("Created MS_permits_2010_2022 (DATE not NULL)")

Created MS_permits_2010_2022 (DATE not NULL)


In [29]:
#2 MS datasets: one with NULL + area of 0.81 ha or greater; one with area of 0.81 ha or greater
import arcpy

gdb = r"C:\Users\Melanie\Desktop\Rx_ClassiFIRE\ClassiFIRE_Rx_burn_permits.gdb"

# Correct, year‑filtered Mississippi permits
ms_2010_2022 = fr"{gdb}\MS_permits_2010_2022"

# Output feature classes
ms_ge081 = fr"{gdb}\MS_permits_ge0_81ha"
ms_null_or_ge081 = fr"{gdb}\MS_permits_NULL_or_ge0_81ha"

# 1) area_ha >= 0.81
arcpy.analysis.Select(
    in_features=ms_2010_2022,
    out_feature_class=ms_ge081,
    where_clause="area_ha >= 0.81"
)
print("Created MS_permits_ge0_81ha")

# 2) area_ha IS NULL OR area_ha >= 0.81
arcpy.analysis.Select(
    in_features=ms_2010_2022,
    out_feature_class=ms_null_or_ge081,
    where_clause="area_ha IS NULL OR area_ha >= 0.81"
)
print("Created MS_permits_NULL_or_ge0_81ha")

Created MS_permits_ge0_81ha
Created MS_permits_NULL_or_ge0_81ha


In [30]:
#Create buffers around each subset
import arcpy

# Output geodatabase (same as NC and AL permit buffers)
gdb_out = r"C:\Users\Melanie\Desktop\Rx_ClassiFIRE\ClassiFIRE_Rx_burn_permits.gdb"

# Mississippi permit subsets (corrected, year-filtered)
datasets = {
    "MS_ge081": fr"{gdb_out}\MS_permits_ge0_81ha",
    "MS_null_or_ge081": fr"{gdb_out}\MS_permits_NULL_or_ge0_81ha"
}

# Buffer distances
buffer_distances = {
    "0_5km": "500 Meters",
    "1km": "1000 Meters",
    "1_5km": "1500 Meters",
    "2km": "2000 Meters",
    "2_5km": "2500 Meters",
    "3km": "3000 Meters",
    "3_5km": "3500 Meters",
    "4km": "4000 Meters"
}

# Create buffers for each MS subset
for label, permits in datasets.items():
    for dist_label, dist in buffer_distances.items():

        out_fc = fr"{gdb_out}\Permits_buf_{dist_label}_{label}"

        arcpy.analysis.Buffer(
            in_features=permits,
            out_feature_class=out_fc,
            buffer_distance_or_field=dist,
            dissolve_option="NONE"
        )

        print(f"Created buffer: {out_fc}")

Created buffer: C:\Users\Melanie\Desktop\Rx_ClassiFIRE\ClassiFIRE_Rx_burn_permits.gdb\Permits_buf_0_5km_MS_ge081
Created buffer: C:\Users\Melanie\Desktop\Rx_ClassiFIRE\ClassiFIRE_Rx_burn_permits.gdb\Permits_buf_1km_MS_ge081
Created buffer: C:\Users\Melanie\Desktop\Rx_ClassiFIRE\ClassiFIRE_Rx_burn_permits.gdb\Permits_buf_1_5km_MS_ge081
Created buffer: C:\Users\Melanie\Desktop\Rx_ClassiFIRE\ClassiFIRE_Rx_burn_permits.gdb\Permits_buf_2km_MS_ge081
Created buffer: C:\Users\Melanie\Desktop\Rx_ClassiFIRE\ClassiFIRE_Rx_burn_permits.gdb\Permits_buf_2_5km_MS_ge081
Created buffer: C:\Users\Melanie\Desktop\Rx_ClassiFIRE\ClassiFIRE_Rx_burn_permits.gdb\Permits_buf_3km_MS_ge081
Created buffer: C:\Users\Melanie\Desktop\Rx_ClassiFIRE\ClassiFIRE_Rx_burn_permits.gdb\Permits_buf_3_5km_MS_ge081
Created buffer: C:\Users\Melanie\Desktop\Rx_ClassiFIRE\ClassiFIRE_Rx_burn_permits.gdb\Permits_buf_4km_MS_ge081
Created buffer: C:\Users\Melanie\Desktop\Rx_ClassiFIRE\ClassiFIRE_Rx_burn_permits.gdb\Permits_buf_0_5km_

In [19]:
#get MS boundary
import arcpy

# Input: national state boundaries
states = r"C:\Users\Melanie\Desktop\Rx_ClassiFIRE\state_boundaries\tl_2025_us_state.shp"

# Output: Mississippi boundary stored with all permit + buffer layers
ms_boundary = r"C:\Users\Melanie\Desktop\Rx_ClassiFIRE\ClassiFIRE_Rx_burn_permits.gdb\MS_boundary"

arcpy.analysis.Select(
    in_features=states,
    out_feature_class=ms_boundary,
    where_clause="STATEFP = '28'"   # 28 = Mississippi FIPS code
)

print("Created Mississippi boundary:", ms_boundary)

Created Mississippi boundary: C:\Users\Melanie\Desktop\Rx_ClassiFIRE\ClassiFIRE_Rx_burn_permits.gdb\MS_boundary


In [20]:
#Clip Rx fire events to MS
import arcpy

# Full SEFM prescribed-fire dataset
events_in = r"C:\Users\Melanie\Desktop\Rx_ClassiFIRE\ClassiFIRE.gdb\SEFM_events_94_22_prescribed"

# Output: Mississippi-only prescribed events
events_out = r"C:\Users\Melanie\Desktop\Rx_ClassiFIRE\ClassiFIRE_Rx_burn_permits.gdb\SEFM_events_94_22_prescribed_MS"

# Mississippi boundary (already created)
ms_boundary = r"C:\Users\Melanie\Desktop\Rx_ClassiFIRE\ClassiFIRE_Rx_burn_permits.gdb\MS_boundary"

arcpy.analysis.Clip(
    in_features=events_in,
    clip_features=ms_boundary,
    out_feature_class=events_out
)

print("Created Mississippi prescribed events subset:", events_out)

Created Mississippi prescribed events subset: C:\Users\Melanie\Desktop\Rx_ClassiFIRE\ClassiFIRE_Rx_burn_permits.gdb\SEFM_events_94_22_prescribed_MS


In [21]:
# only include MS Rx events 2010-22
import arcpy

# Input: MS prescribed events (already clipped to MS)
in_fc = r"C:\Users\Melanie\Desktop\Rx_ClassiFIRE\ClassiFIRE_Rx_burn_permits.gdb\SEFM_events_94_22_prescribed_MS"

# Output: MS prescribed events limited to 2010–2022
out_fc = r"C:\Users\Melanie\Desktop\Rx_ClassiFIRE\ClassiFIRE_Rx_burn_permits.gdb\SEFM_events_2010_2022_prescribed_MS"

# Keep only events from 2010 through 2022
where = "event_year >= 2010 AND event_year <= 2022"

arcpy.analysis.Select(
    in_features=in_fc,
    out_feature_class=out_fc,
    where_clause=where
)

print("MS prescribed events (2010–2022) subset created.")

MS prescribed events (2010–2022) subset created.


In [31]:
#Add buffer fields to MS permit data
import arcpy

gdb = r"C:\Users\Melanie\Desktop\Rx_ClassiFIRE\ClassiFIRE_Rx_burn_permits.gdb"

# Mississippi permit datasets (corrected, year-filtered)
permit_layers = [
    fr"{gdb}\MS_permits_ge0_81ha",
    fr"{gdb}\MS_permits_NULL_or_ge0_81ha"
]

# Buffer labels (same as NC and AL)
buffer_labels = [
    "0_5km",
    "1km",
    "1_5km",
    "2km",
    "2_5km",
    "3km",
    "3_5km",
    "4km"
]

# Add fields to each permit layer
for permits in permit_layers:
    existing_fields = [f.name for f in arcpy.ListFields(permits)]
    
    for label in buffer_labels:
        field = f"buffer_{label}"
        
        if field not in existing_fields:
            arcpy.management.AddField(permits, field, "TEXT")
            print(f"Added field {field} to {permits}")
        else:
            print(f"Field {field} already exists in {permits}")


Added field buffer_0_5km to C:\Users\Melanie\Desktop\Rx_ClassiFIRE\ClassiFIRE_Rx_burn_permits.gdb\MS_permits_ge0_81ha
Added field buffer_1km to C:\Users\Melanie\Desktop\Rx_ClassiFIRE\ClassiFIRE_Rx_burn_permits.gdb\MS_permits_ge0_81ha
Added field buffer_1_5km to C:\Users\Melanie\Desktop\Rx_ClassiFIRE\ClassiFIRE_Rx_burn_permits.gdb\MS_permits_ge0_81ha
Added field buffer_2km to C:\Users\Melanie\Desktop\Rx_ClassiFIRE\ClassiFIRE_Rx_burn_permits.gdb\MS_permits_ge0_81ha
Added field buffer_2_5km to C:\Users\Melanie\Desktop\Rx_ClassiFIRE\ClassiFIRE_Rx_burn_permits.gdb\MS_permits_ge0_81ha
Added field buffer_3km to C:\Users\Melanie\Desktop\Rx_ClassiFIRE\ClassiFIRE_Rx_burn_permits.gdb\MS_permits_ge0_81ha
Added field buffer_3_5km to C:\Users\Melanie\Desktop\Rx_ClassiFIRE\ClassiFIRE_Rx_burn_permits.gdb\MS_permits_ge0_81ha
Added field buffer_4km to C:\Users\Melanie\Desktop\Rx_ClassiFIRE\ClassiFIRE_Rx_burn_permits.gdb\MS_permits_ge0_81ha
Added field buffer_0_5km to C:\Users\Melanie\Desktop\Rx_ClassiFI

In [32]:
#MS spatial joins
import arcpy

gdb = r"C:\Users\Melanie\Desktop\Rx_ClassiFIRE\ClassiFIRE_Rx_burn_permits.gdb"

# Mississippi prescribed events (clipped + year-filtered)
events = fr"{gdb}\SEFM_events_2010_2022_prescribed_MS"

# Two Mississippi permit datasets (corrected, year-filtered)
permit_sets = [
    "MS_ge081",
    "MS_null_or_ge081"
]

# Buffer labels (same as NC and AL)
buffer_labels = [
    "0_5km",
    "1km",
    "1_5km",
    "2km",
    "2_5km",
    "3km",
    "3_5km",
    "4km"
]

# Spatial join for each permit set × each buffer distance
for pset in permit_sets:
    for label in buffer_labels:

        permits_buf = fr"{gdb}\Permits_buf_{label}_{pset}"
        sj = fr"{gdb}\PermitEventJoin_{label}_{pset}"

        arcpy.analysis.SpatialJoin(
            target_features=permits_buf,
            join_features=events,
            out_feature_class=sj,
            join_operation="JOIN_ONE_TO_MANY",
            match_option="INTERSECT"
        )

        print(f"Spatial join completed: {sj}")

Spatial join completed: C:\Users\Melanie\Desktop\Rx_ClassiFIRE\ClassiFIRE_Rx_burn_permits.gdb\PermitEventJoin_0_5km_MS_ge081
Spatial join completed: C:\Users\Melanie\Desktop\Rx_ClassiFIRE\ClassiFIRE_Rx_burn_permits.gdb\PermitEventJoin_1km_MS_ge081
Spatial join completed: C:\Users\Melanie\Desktop\Rx_ClassiFIRE\ClassiFIRE_Rx_burn_permits.gdb\PermitEventJoin_1_5km_MS_ge081
Spatial join completed: C:\Users\Melanie\Desktop\Rx_ClassiFIRE\ClassiFIRE_Rx_burn_permits.gdb\PermitEventJoin_2km_MS_ge081
Spatial join completed: C:\Users\Melanie\Desktop\Rx_ClassiFIRE\ClassiFIRE_Rx_burn_permits.gdb\PermitEventJoin_2_5km_MS_ge081
Spatial join completed: C:\Users\Melanie\Desktop\Rx_ClassiFIRE\ClassiFIRE_Rx_burn_permits.gdb\PermitEventJoin_3km_MS_ge081
Spatial join completed: C:\Users\Melanie\Desktop\Rx_ClassiFIRE\ClassiFIRE_Rx_burn_permits.gdb\PermitEventJoin_3_5km_MS_ge081
Spatial join completed: C:\Users\Melanie\Desktop\Rx_ClassiFIRE\ClassiFIRE_Rx_burn_permits.gdb\PermitEventJoin_4km_MS_ge081
Spatial 

In [33]:
#MS temporal matching
import arcpy
from datetime import datetime

# Paths
gdb = r"C:\Users\Melanie\Desktop\Rx_ClassiFIRE\ClassiFIRE_Rx_burn_permits.gdb"

# Mississippi permit datasets (corrected, year-filtered)
permit_sets = {
    "MS_ge081": fr"{gdb}\MS_permits_ge0_81ha",
    "MS_null_or_ge081": fr"{gdb}\MS_permits_NULL_or_ge0_81ha"
}

# Mississippi prescribed events (clipped + year-filtered)
events = fr"{gdb}\SEFM_events_2010_2022_prescribed_MS"

# Buffer labels
buffer_labels = [
    "0_5km",
    "1km",
    "1_5km",
    "2km",
    "2_5km",
    "3km",
    "3_5km",
    "4km"
]

# --- DATE PARSERS ---

# SEFM event dates: YYYYMMDD integer
def parse_sefm_date(val):
    s = str(val)
    year = int(s[0:4])
    month = int(s[4:6])
    day = int(s[6:8])
    return datetime(year, month, day)

# Permit dates: MM/DD/YYYY or datetime
def parse_permit_date(val):
    if isinstance(val, datetime):
        return val
    return datetime.strptime(val, "%m/%d/%Y")

# --- MAIN LOOP FOR BOTH MISSISSIPPI PERMIT DATASETS ---

for pset_label, permits_fc in permit_sets.items():

    print(f"\n=== Processing permit set: {pset_label} ===")

    for label in buffer_labels:

        print(f"  Buffer {label}...")

        # Spatial join table for this permit set + buffer
        sj = fr"{gdb}\PermitEventJoin_{label}_{pset_label}"

        # Build mapping: permit OBJECTID → list of (event_start, event_end)
        permit_to_events = {}

        with arcpy.da.SearchCursor(
            sj,
            ["TARGET_FID", "MIN_prebd_min_corrected", "MAX_bd_min_corrected_plus8"]
        ) as cur:
            for pid, tmin_raw, tmax_raw in cur:
                if tmin_raw is None or tmax_raw is None:
                    continue
                tmin = parse_sefm_date(tmin_raw)
                tmax = parse_sefm_date(tmax_raw)
                permit_to_events.setdefault(pid, []).append((tmin, tmax))

        # Field to update on the permit layer
        field = f"buffer_{label}"

        with arcpy.da.UpdateCursor(
            permits_fc,
            ["OBJECTID", "DATE", field]
        ) as cur:

            for pid, permit_date_raw, _ in cur:

                permit_date = parse_permit_date(permit_date_raw)

                # Default: no spatial match → no_match
                classification = "no_match"

                # Only check temporal overlap if spatial matches exist
                if pid in permit_to_events:
                    for (tmin, tmax) in permit_to_events[pid]:
                        if tmin <= permit_date <= tmax:
                            classification = "match"
                            break

                cur.updateRow([pid, permit_date_raw, classification])

        print(f"    Completed buffer {label}")

print("\nAll temporal matching complete for Mississippi.")


=== Processing permit set: MS_ge081 ===
  Buffer 0_5km...
    Completed buffer 0_5km
  Buffer 1km...
    Completed buffer 1km
  Buffer 1_5km...
    Completed buffer 1_5km
  Buffer 2km...
    Completed buffer 2km
  Buffer 2_5km...
    Completed buffer 2_5km
  Buffer 3km...
    Completed buffer 3km
  Buffer 3_5km...
    Completed buffer 3_5km
  Buffer 4km...
    Completed buffer 4km

=== Processing permit set: MS_null_or_ge081 ===
  Buffer 0_5km...
    Completed buffer 0_5km
  Buffer 1km...
    Completed buffer 1km
  Buffer 1_5km...
    Completed buffer 1_5km
  Buffer 2km...
    Completed buffer 2km
  Buffer 2_5km...
    Completed buffer 2_5km
  Buffer 3km...
    Completed buffer 3km
  Buffer 3_5km...
    Completed buffer 3_5km
  Buffer 4km...
    Completed buffer 4km

All temporal matching complete for Mississippi.


In [34]:
#Calculate % permits matching for MS datasets
import arcpy

gdb = r"C:\Users\Melanie\Desktop\Rx_ClassiFIRE\ClassiFIRE_Rx_burn_permits.gdb"

# Two Mississippi permit datasets (corrected, year-filtered)
permit_sets = {
    "MS_ge081": fr"{gdb}\MS_permits_ge0_81ha",
    "MS_null_or_ge081": fr"{gdb}\MS_permits_NULL_or_ge0_81ha"
}

buffer_labels = [
    "0_5km",
    "1km",
    "1_5km",
    "2km",
    "2_5km",
    "3km",
    "3_5km",
    "4km"
]

for label, permits_fc in permit_sets.items():

    print(f"\n=== Match percentages for {label} ===")

    total = int(arcpy.management.GetCount(permits_fc)[0])
    results = {}

    for buf in buffer_labels:
        field = f"buffer_{buf}"
        match_count = 0

        with arcpy.da.SearchCursor(permits_fc, [field]) as cur:
            for (val,) in cur:
                if val == "match":
                    match_count += 1

        pct = (match_count / total) * 100
        results[buf] = pct
        print(f"{buf}: {pct:.2f}% match")

    print("Summary:", results)


=== Match percentages for MS_ge081 ===
0_5km: 15.49% match
1km: 25.05% match
1_5km: 31.16% match
2km: 35.86% match
2_5km: 39.96% match
3km: 44.21% match
3_5km: 48.36% match
4km: 52.10% match
Summary: {'0_5km': 15.48557819528976, '1km': 25.0542471553321, '1_5km': 31.15639057951839, '2km': 35.85869277586663, '2_5km': 39.95501455411484, '3km': 44.21275469700979, '3_5km': 48.364646731939665, '4km': 52.095792537708384}

=== Match percentages for MS_null_or_ge081 ===
0_5km: 14.63% match
1km: 23.84% match
1_5km: 29.84% match
2km: 34.61% match
2_5km: 38.77% match
3km: 43.11% match
3_5km: 47.32% match
4km: 51.09% match
Summary: {'0_5km': 14.629543393544274, '1km': 23.840101958285338, '1_5km': 29.84485674370726, '2km': 34.606994926594936, '2_5km': 38.771108551260994, '3km': 43.10923751868827, '3_5km': 47.32236954976594, '4km': 51.09188500281856}


In [35]:
#what % of MS events match (exclude NULL areas here)
import arcpy

gdb = r"C:\Users\Melanie\Desktop\Rx_ClassiFIRE\ClassiFIRE_Rx_burn_permits.gdb"
events_fc = fr"{gdb}\SEFM_events_2010_2022_prescribed_MS"

buffer_labels = [
    "0_5km", "1km", "1_5km", "2km",
    "2_5km", "3km", "3_5km", "4km"
]

existing = [f.name for f in arcpy.ListFields(events_fc)]

for label in buffer_labels:
    field = f"evt_buf_{label}"
    if field not in existing:
        arcpy.management.AddField(events_fc, field, "TEXT")
        print(f"Added field {field}")

Added field evt_buf_0_5km
Added field evt_buf_1km
Added field evt_buf_1_5km
Added field evt_buf_2km
Added field evt_buf_2_5km
Added field evt_buf_3km
Added field evt_buf_3_5km
Added field evt_buf_4km


In [36]:
import arcpy
from datetime import datetime

gdb = r"C:\Users\Melanie\Desktop\Rx_ClassiFIRE\ClassiFIRE_Rx_burn_permits.gdb"
events_fc = fr"{gdb}\SEFM_events_2010_2022_prescribed_MS"

buffer_labels = [
    "0_5km", "1km", "1_5km", "2km",
    "2_5km", "3km", "3_5km", "4km"
]

def parse_sefm_date(val):
    s = str(val)
    return datetime(int(s[0:4]), int(s[4:6]), int(s[6:8]))

def parse_permit_date(val):
    if isinstance(val, datetime):
        return val
    return datetime.strptime(val, "%m/%d/%Y")

for label in buffer_labels:

    print(f"Processing event matching for buffer {label}...")

    # Use MS_ge081 (parallel to AL_big)
    sj = fr"{gdb}\PermitEventJoin_{label}_MS_ge081"

    event_to_permits = {}

    with arcpy.da.SearchCursor(
        sj,
        ["JOIN_FID", "DATE"]
    ) as cur:
        for eid, permit_date_raw in cur:
            if permit_date_raw is None:
                continue
            permit_date = parse_permit_date(permit_date_raw)
            event_to_permits.setdefault(eid, []).append(permit_date)

    field = f"evt_buf_{label}"

    with arcpy.da.UpdateCursor(
        events_fc,
        ["OBJECTID", "MIN_prebd_min_corrected", "MAX_bd_min_corrected_plus8", field]
    ) as cur:

        for eid, tmin_raw, tmax_raw, _ in cur:

            if tmin_raw is None or tmax_raw is None:
                cur.updateRow([eid, tmin_raw, tmax_raw, "no_match"])
                continue

            tmin = parse_sefm_date(tmin_raw)
            tmax = parse_sefm_date(tmax_raw)

            classification = "no_match"

            if eid in event_to_permits:
                for permit_date in event_to_permits[eid]:
                    if tmin <= permit_date <= tmax:
                        classification = "match"
                        break

            cur.updateRow([eid, tmin_raw, tmax_raw, classification])

    print(f"Completed buffer {label}")

print("All Mississippi event matching complete.")

Processing event matching for buffer 0_5km...
Completed buffer 0_5km
Processing event matching for buffer 1km...
Completed buffer 1km
Processing event matching for buffer 1_5km...
Completed buffer 1_5km
Processing event matching for buffer 2km...
Completed buffer 2km
Processing event matching for buffer 2_5km...
Completed buffer 2_5km
Processing event matching for buffer 3km...
Completed buffer 3km
Processing event matching for buffer 3_5km...
Completed buffer 3_5km
Processing event matching for buffer 4km...
Completed buffer 4km
All Mississippi event matching complete.


In [37]:
import arcpy

gdb = r"C:\Users\Melanie\Desktop\Rx_ClassiFIRE\ClassiFIRE_Rx_burn_permits.gdb"
events_fc = fr"{gdb}\SEFM_events_2010_2022_prescribed_MS"

buffer_labels = [
    "0_5km", "1km", "1_5km", "2km",
    "2_5km", "3km", "3_5km", "4km"
]

total = int(arcpy.management.GetCount(events_fc)[0])
results = {}

for label in buffer_labels:
    field = f"evt_buf_{label}"
    match_count = 0

    with arcpy.da.SearchCursor(events_fc, [field]) as cur:
        for (val,) in cur:
            if val == "match":
                match_count += 1

    pct = (match_count / total) * 100
    results[label] = pct
    print(f"{label}: {pct:.2f}% matched")

print("\nSummary:", results)

0_5km: 7.52% matched
1km: 13.38% matched
1_5km: 17.24% matched
2km: 20.00% matched
2_5km: 22.33% matched
3km: 24.43% matched
3_5km: 26.38% matched
4km: 28.40% matched

Summary: {'0_5km': 7.524454477050415, '1km': 13.384123401053424, '1_5km': 17.24040632054176, '2km': 20.0042996882726, '2_5km': 22.334193270987853, '3km': 24.432978609050842, '3_5km': 26.377243899817266, '4km': 28.396753735354185}


In [38]:
#Now Florida. First, copy permit data over to permit gdb:
import arcpy

gdb_in  = r"C:\Users\Melanie\Desktop\Rx_ClassiFIRE\SE_BurnData2010_2023.gdb"
gdb_out = r"C:\Users\Melanie\Desktop\Rx_ClassiFIRE\ClassiFIRE_Rx_burn_permits.gdb"

src = fr"{gdb_in}\BurnData_FL"
dst = fr"{gdb_out}\BurnData_FL"

arcpy.management.CopyFeatures(src, dst)

print("Copied BurnData_FL into permit geodatabase.")

Copied BurnData_FL into permit geodatabase.


In [39]:
#Only use FL permits 2010-2022 to match events (no NULL date in FL)
import arcpy

gdb = r"C:\Users\Melanie\Desktop\Rx_ClassiFIRE\ClassiFIRE_Rx_burn_permits.gdb"

fl_permits = fr"{gdb}\BurnData_FL"
fl_2010_2022 = fr"{gdb}\FL_permits_2010_2022"

# Keep only permits from 2010–2022
where = "YEAR >= 2010 AND YEAR <= 2022"

arcpy.analysis.Select(
    in_features=fl_permits,
    out_feature_class=fl_2010_2022,
    where_clause=where
)

print("Created FL_permits_2010_2022")

Created FL_permits_2010_2022


In [40]:
#2 FL datasets: one with NULL + area of 0.81 ha or greater; one with area of 0.81 ha or greater
import arcpy

gdb = r"C:\Users\Melanie\Desktop\Rx_ClassiFIRE\ClassiFIRE_Rx_burn_permits.gdb"

# Correct, year‑filtered Florida permits
fl_2010_2022 = fr"{gdb}\FL_permits_2010_2022"

# Output feature classes
fl_ge081 = fr"{gdb}\FL_permits_ge0_81ha"
fl_null_or_ge081 = fr"{gdb}\FL_permits_NULL_or_ge0_81ha"

# 1) area_ha >= 0.81
arcpy.analysis.Select(
    in_features=fl_2010_2022,
    out_feature_class=fl_ge081,
    where_clause="area_ha >= 0.81"
)
print("Created FL_permits_ge0_81ha")

# 2) area_ha IS NULL OR area_ha >= 0.81
arcpy.analysis.Select(
    in_features=fl_2010_2022,
    out_feature_class=fl_null_or_ge081,
    where_clause="area_ha IS NULL OR area_ha >= 0.81"
)
print("Created FL_permits_NULL_or_ge0_81ha")

Created FL_permits_ge0_81ha
Created FL_permits_NULL_or_ge0_81ha


In [41]:
#Create buffers around each subset
import arcpy

# Output geodatabase (same as NC, AL, MS)
gdb_out = r"C:\Users\Melanie\Desktop\Rx_ClassiFIRE\ClassiFIRE_Rx_burn_permits.gdb"

# Florida permit subsets (year-filtered)
datasets = {
    "FL_ge081": fr"{gdb_out}\FL_permits_ge0_81ha",
    "FL_null_or_ge081": fr"{gdb_out}\FL_permits_NULL_or_ge0_81ha"
}

# Buffer distances
buffer_distances = {
    "0_5km": "500 Meters",
    "1km": "1000 Meters",
    "1_5km": "1500 Meters",
    "2km": "2000 Meters",
    "2_5km": "2500 Meters",
    "3km": "3000 Meters",
    "3_5km": "3500 Meters",
    "4km": "4000 Meters"
}

# Create buffers for each FL subset
for label, permits in datasets.items():
    for dist_label, dist in buffer_distances.items():

        out_fc = fr"{gdb_out}\Permits_buf_{dist_label}_{label}"

        arcpy.analysis.Buffer(
            in_features=permits,
            out_feature_class=out_fc,
            buffer_distance_or_field=dist,
            dissolve_option="NONE"
        )

        print(f"Created buffer: {out_fc}")

Created buffer: C:\Users\Melanie\Desktop\Rx_ClassiFIRE\ClassiFIRE_Rx_burn_permits.gdb\Permits_buf_0_5km_FL_ge081
Created buffer: C:\Users\Melanie\Desktop\Rx_ClassiFIRE\ClassiFIRE_Rx_burn_permits.gdb\Permits_buf_1km_FL_ge081
Created buffer: C:\Users\Melanie\Desktop\Rx_ClassiFIRE\ClassiFIRE_Rx_burn_permits.gdb\Permits_buf_1_5km_FL_ge081
Created buffer: C:\Users\Melanie\Desktop\Rx_ClassiFIRE\ClassiFIRE_Rx_burn_permits.gdb\Permits_buf_2km_FL_ge081
Created buffer: C:\Users\Melanie\Desktop\Rx_ClassiFIRE\ClassiFIRE_Rx_burn_permits.gdb\Permits_buf_2_5km_FL_ge081
Created buffer: C:\Users\Melanie\Desktop\Rx_ClassiFIRE\ClassiFIRE_Rx_burn_permits.gdb\Permits_buf_3km_FL_ge081
Created buffer: C:\Users\Melanie\Desktop\Rx_ClassiFIRE\ClassiFIRE_Rx_burn_permits.gdb\Permits_buf_3_5km_FL_ge081
Created buffer: C:\Users\Melanie\Desktop\Rx_ClassiFIRE\ClassiFIRE_Rx_burn_permits.gdb\Permits_buf_4km_FL_ge081
Created buffer: C:\Users\Melanie\Desktop\Rx_ClassiFIRE\ClassiFIRE_Rx_burn_permits.gdb\Permits_buf_0_5km_

In [42]:
#create FL boundary
import arcpy

# Input: national state boundaries
states = r"C:\Users\Melanie\Desktop\Rx_ClassiFIRE\state_boundaries\tl_2025_us_state.shp"

# Output: Florida boundary stored with all permit + buffer layers
fl_boundary = r"C:\Users\Melanie\Desktop\Rx_ClassiFIRE\ClassiFIRE_Rx_burn_permits.gdb\FL_boundary"

arcpy.analysis.Select(
    in_features=states,
    out_feature_class=fl_boundary,
    where_clause="STATEFP = '12'"   # 12 = Florida FIPS code
)

print("Created Florida boundary:", fl_boundary)

Created Florida boundary: C:\Users\Melanie\Desktop\Rx_ClassiFIRE\ClassiFIRE_Rx_burn_permits.gdb\FL_boundary


In [43]:
#Clip Rx events to FL
import arcpy

# Full SEFM prescribed-fire dataset
events_in = r"C:\Users\Melanie\Desktop\Rx_ClassiFIRE\ClassiFIRE.gdb\SEFM_events_94_22_prescribed"

# Output: Florida-only prescribed events
events_out = r"C:\Users\Melanie\Desktop\Rx_ClassiFIRE\ClassiFIRE_Rx_burn_permits.gdb\SEFM_events_94_22_prescribed_FL"

# Florida boundary (already created)
fl_boundary = r"C:\Users\Melanie\Desktop\Rx_ClassiFIRE\ClassiFIRE_Rx_burn_permits.gdb\FL_boundary"

arcpy.analysis.Clip(
    in_features=events_in,
    clip_features=fl_boundary,
    out_feature_class=events_out
)

print("Created Florida prescribed events subset:", events_out)

Created Florida prescribed events subset: C:\Users\Melanie\Desktop\Rx_ClassiFIRE\ClassiFIRE_Rx_burn_permits.gdb\SEFM_events_94_22_prescribed_FL


In [44]:
# only include FL Rx events 2010-22
import arcpy

# Input: FL prescribed events (already clipped to FL)
in_fc = r"C:\Users\Melanie\Desktop\Rx_ClassiFIRE\ClassiFIRE_Rx_burn_permits.gdb\SEFM_events_94_22_prescribed_FL"

# Output: FL prescribed events limited to 2010–2022
out_fc = r"C:\Users\Melanie\Desktop\Rx_ClassiFIRE\ClassiFIRE_Rx_burn_permits.gdb\SEFM_events_2010_2022_prescribed_FL"

# Keep only events from 2010 through 2022
where = "event_year >= 2010 AND event_year <= 2022"

arcpy.analysis.Select(
    in_features=in_fc,
    out_feature_class=out_fc,
    where_clause=where
)

print("FL prescribed events (2010–2022) subset created.")

FL prescribed events (2010–2022) subset created.


In [45]:
#Add buffer fields to FL permit data
import arcpy

gdb = r"C:\Users\Melanie\Desktop\Rx_ClassiFIRE\ClassiFIRE_Rx_burn_permits.gdb"

# Florida permit datasets (year‑filtered)
permit_layers = [
    fr"{gdb}\FL_permits_ge0_81ha",
    fr"{gdb}\FL_permits_NULL_or_ge0_81ha"
]

# Buffer labels (same across all states)
buffer_labels = [
    "0_5km",
    "1km",
    "1_5km",
    "2km",
    "2_5km",
    "3km",
    "3_5km",
    "4km"
]

# Add fields to each permit layer
for permits in permit_layers:
    existing_fields = [f.name for f in arcpy.ListFields(permits)]
    
    for label in buffer_labels:
        field = f"buffer_{label}"
        
        if field not in existing_fields:
            arcpy.management.AddField(permits, field, "TEXT")
            print(f"Added field {field} to {permits}")
        else:
            print(f"Field {field} already exists in {permits}")

Added field buffer_0_5km to C:\Users\Melanie\Desktop\Rx_ClassiFIRE\ClassiFIRE_Rx_burn_permits.gdb\FL_permits_ge0_81ha
Added field buffer_1km to C:\Users\Melanie\Desktop\Rx_ClassiFIRE\ClassiFIRE_Rx_burn_permits.gdb\FL_permits_ge0_81ha
Added field buffer_1_5km to C:\Users\Melanie\Desktop\Rx_ClassiFIRE\ClassiFIRE_Rx_burn_permits.gdb\FL_permits_ge0_81ha
Added field buffer_2km to C:\Users\Melanie\Desktop\Rx_ClassiFIRE\ClassiFIRE_Rx_burn_permits.gdb\FL_permits_ge0_81ha
Added field buffer_2_5km to C:\Users\Melanie\Desktop\Rx_ClassiFIRE\ClassiFIRE_Rx_burn_permits.gdb\FL_permits_ge0_81ha
Added field buffer_3km to C:\Users\Melanie\Desktop\Rx_ClassiFIRE\ClassiFIRE_Rx_burn_permits.gdb\FL_permits_ge0_81ha
Added field buffer_3_5km to C:\Users\Melanie\Desktop\Rx_ClassiFIRE\ClassiFIRE_Rx_burn_permits.gdb\FL_permits_ge0_81ha
Added field buffer_4km to C:\Users\Melanie\Desktop\Rx_ClassiFIRE\ClassiFIRE_Rx_burn_permits.gdb\FL_permits_ge0_81ha
Added field buffer_0_5km to C:\Users\Melanie\Desktop\Rx_ClassiFI

In [46]:
#FL spatial joins
import arcpy

gdb = r"C:\Users\Melanie\Desktop\Rx_ClassiFIRE\ClassiFIRE_Rx_burn_permits.gdb"

# Florida prescribed events (clipped + year-filtered)
events = fr"{gdb}\SEFM_events_2010_2022_prescribed_FL"

# Two Florida permit datasets (year-filtered)
permit_sets = [
    "FL_ge081",
    "FL_null_or_ge081"
]

# Buffer labels (same as NC, AL, MS)
buffer_labels = [
    "0_5km",
    "1km",
    "1_5km",
    "2km",
    "2_5km",
    "3km",
    "3_5km",
    "4km"
]

# Spatial join for each permit set × each buffer distance
for pset in permit_sets:
    for label in buffer_labels:

        permits_buf = fr"{gdb}\Permits_buf_{label}_{pset}"
        sj = fr"{gdb}\PermitEventJoin_{label}_{pset}"

        arcpy.analysis.SpatialJoin(
            target_features=permits_buf,
            join_features=events,
            out_feature_class=sj,
            join_operation="JOIN_ONE_TO_MANY",
            match_option="INTERSECT"
        )

        print(f"Spatial join completed: {sj}")

Spatial join completed: C:\Users\Melanie\Desktop\Rx_ClassiFIRE\ClassiFIRE_Rx_burn_permits.gdb\PermitEventJoin_0_5km_FL_ge081
Spatial join completed: C:\Users\Melanie\Desktop\Rx_ClassiFIRE\ClassiFIRE_Rx_burn_permits.gdb\PermitEventJoin_1km_FL_ge081
Spatial join completed: C:\Users\Melanie\Desktop\Rx_ClassiFIRE\ClassiFIRE_Rx_burn_permits.gdb\PermitEventJoin_1_5km_FL_ge081
Spatial join completed: C:\Users\Melanie\Desktop\Rx_ClassiFIRE\ClassiFIRE_Rx_burn_permits.gdb\PermitEventJoin_2km_FL_ge081
Spatial join completed: C:\Users\Melanie\Desktop\Rx_ClassiFIRE\ClassiFIRE_Rx_burn_permits.gdb\PermitEventJoin_2_5km_FL_ge081
Spatial join completed: C:\Users\Melanie\Desktop\Rx_ClassiFIRE\ClassiFIRE_Rx_burn_permits.gdb\PermitEventJoin_3km_FL_ge081
Spatial join completed: C:\Users\Melanie\Desktop\Rx_ClassiFIRE\ClassiFIRE_Rx_burn_permits.gdb\PermitEventJoin_3_5km_FL_ge081
Spatial join completed: C:\Users\Melanie\Desktop\Rx_ClassiFIRE\ClassiFIRE_Rx_burn_permits.gdb\PermitEventJoin_4km_FL_ge081
Spatial 

In [47]:
# FL temporal matching
import arcpy
from datetime import datetime

# Paths
gdb = r"C:\Users\Melanie\Desktop\Rx_ClassiFIRE\ClassiFIRE_Rx_burn_permits.gdb"

# Florida permit datasets (year-filtered)
permit_sets = {
    "FL_ge081": fr"{gdb}\FL_permits_ge0_81ha",
    "FL_null_or_ge081": fr"{gdb}\FL_permits_NULL_or_ge0_81ha"
}

# Florida prescribed events (clipped + year-filtered)
events = fr"{gdb}\SEFM_events_2010_2022_prescribed_FL"

# Buffer labels
buffer_labels = [
    "0_5km",
    "1km",
    "1_5km",
    "2km",
    "2_5km",
    "3km",
    "3_5km",
    "4km"
]

# --- DATE PARSERS ---

# SEFM event dates: YYYYMMDD integer
def parse_sefm_date(val):
    s = str(val)
    year = int(s[0:4])
    month = int(s[4:6])
    day = int(s[6:8])
    return datetime(year, month, day)

# Permit dates: MM/DD/YYYY or datetime
def parse_permit_date(val):
    if isinstance(val, datetime):
        return val
    return datetime.strptime(val, "%m/%d/%Y")

# --- MAIN LOOP FOR BOTH FLORIDA PERMIT DATASETS ---

for pset_label, permits_fc in permit_sets.items():

    print(f"\n=== Processing permit set: {pset_label} ===")

    for label in buffer_labels:

        print(f"  Buffer {label}...")

        # Spatial join table for this permit set + buffer
        sj = fr"{gdb}\PermitEventJoin_{label}_{pset_label}"

        # Build mapping: permit OBJECTID → list of (event_start, event_end)
        permit_to_events = {}

        with arcpy.da.SearchCursor(
            sj,
            ["TARGET_FID", "MIN_prebd_min_corrected", "MAX_bd_min_corrected_plus8"]
        ) as cur:
            for pid, tmin_raw, tmax_raw in cur:
                if tmin_raw is None or tmax_raw is None:
                    continue
                tmin = parse_sefm_date(tmin_raw)
                tmax = parse_sefm_date(tmax_raw)
                permit_to_events.setdefault(pid, []).append((tmin, tmax))

        # Field to update on the permit layer
        field = f"buffer_{label}"

        with arcpy.da.UpdateCursor(
            permits_fc,
            ["OBJECTID", "DATE", field]
        ) as cur:

            for pid, permit_date_raw, _ in cur:

                permit_date = parse_permit_date(permit_date_raw)

                # Default: no spatial match → no_match
                classification = "no_match"

                # Only check temporal overlap if spatial matches exist
                if pid in permit_to_events:
                    for (tmin, tmax) in permit_to_events[pid]:
                        if tmin <= permit_date <= tmax:
                            classification = "match"
                            break

                cur.updateRow([pid, permit_date_raw, classification])

        print(f"    Completed buffer {label}")

print("\nAll temporal matching complete for Florida.")


=== Processing permit set: FL_ge081 ===
  Buffer 0_5km...
    Completed buffer 0_5km
  Buffer 1km...
    Completed buffer 1km
  Buffer 1_5km...
    Completed buffer 1_5km
  Buffer 2km...
    Completed buffer 2km
  Buffer 2_5km...
    Completed buffer 2_5km
  Buffer 3km...
    Completed buffer 3km
  Buffer 3_5km...
    Completed buffer 3_5km
  Buffer 4km...
    Completed buffer 4km

=== Processing permit set: FL_null_or_ge081 ===
  Buffer 0_5km...
    Completed buffer 0_5km
  Buffer 1km...
    Completed buffer 1km
  Buffer 1_5km...
    Completed buffer 1_5km
  Buffer 2km...
    Completed buffer 2km
  Buffer 2_5km...
    Completed buffer 2_5km
  Buffer 3km...
    Completed buffer 3km
  Buffer 3_5km...
    Completed buffer 3_5km
  Buffer 4km...
    Completed buffer 4km

All temporal matching complete for Florida.


In [48]:
# FL permits % matching
import arcpy

gdb = r"C:\Users\Melanie\Desktop\Rx_ClassiFIRE\ClassiFIRE_Rx_burn_permits.gdb"

# Two Florida permit datasets (year‑filtered)
permit_sets = {
    "FL_ge081": fr"{gdb}\FL_permits_ge0_81ha",
    "FL_null_or_ge081": fr"{gdb}\FL_permits_NULL_or_ge0_81ha"
}

buffer_labels = [
    "0_5km",
    "1km",
    "1_5km",
    "2km",
    "2_5km",
    "3km",
    "3_5km",
    "4km"
]

for label, permits_fc in permit_sets.items():

    print(f"\n=== Match percentages for {label} ===")

    total = int(arcpy.management.GetCount(permits_fc)[0])
    results = {}

    for buf in buffer_labels:
        field = f"buffer_{buf}"
        match_count = 0

        with arcpy.da.SearchCursor(permits_fc, [field]) as cur:
            for (val,) in cur:
                if val == "match":
                    match_count += 1

        pct = (match_count / total) * 100
        results[buf] = pct
        print(f"{buf}: {pct:.2f}% match")

    print("Summary:", results)


=== Match percentages for FL_ge081 ===
0_5km: 39.95% match
1km: 52.24% match
1_5km: 60.20% match
2km: 66.27% match
2_5km: 71.49% match
3km: 75.70% match
3_5km: 79.28% match
4km: 82.30% match
Summary: {'0_5km': 39.94937399323282, '1km': 52.238538834579096, '1_5km': 60.201907424295364, '2km': 66.26807919475671, '2_5km': 71.48920575125074, '3km': 75.70420434497277, '3_5km': 79.28041182637155, '4km': 82.30305716306859}

=== Match percentages for FL_null_or_ge081 ===
0_5km: 13.68% match
1km: 19.63% match
1_5km: 24.47% match
2km: 29.01% match
2_5km: 33.45% match
3km: 37.81% match
3_5km: 41.93% match
4km: 45.93% match
Summary: {'0_5km': 13.679489759551549, '1km': 19.631732854542573, '1_5km': 24.465435886596744, '2km': 29.0079519890091, '2_5km': 33.44815917019595, '3km': 37.80753106087605, '3_5km': 41.92765747422883, '4km': 45.92849843107583}


In [49]:
#Create fields for FL event matching. Include both subsets of data here
import arcpy

gdb = r"C:\Users\Melanie\Desktop\Rx_ClassiFIRE\ClassiFIRE_Rx_burn_permits.gdb"
events_fc = fr"{gdb}\SEFM_events_2010_2022_prescribed_FL"

buffer_labels = [
    "0_5km", "1km", "1_5km", "2km",
    "2_5km", "3km", "3_5km", "4km"
]

# Two permit subsets → two sets of event fields
suffixes = [
    "ge081",
    "null_or_ge081"
]

existing = [f.name for f in arcpy.ListFields(events_fc)]

for suf in suffixes:
    for label in buffer_labels:
        field = f"evt_buf_{label}_{suf}"
        if field not in existing:
            arcpy.management.AddField(events_fc, field, "TEXT")
            print(f"Added field {field}")
        else:
            print(f"Field {field} already exists")

Added field evt_buf_0_5km_ge081
Added field evt_buf_1km_ge081
Added field evt_buf_1_5km_ge081
Added field evt_buf_2km_ge081
Added field evt_buf_2_5km_ge081
Added field evt_buf_3km_ge081
Added field evt_buf_3_5km_ge081
Added field evt_buf_4km_ge081
Added field evt_buf_0_5km_null_or_ge081
Added field evt_buf_1km_null_or_ge081
Added field evt_buf_1_5km_null_or_ge081
Added field evt_buf_2km_null_or_ge081
Added field evt_buf_2_5km_null_or_ge081
Added field evt_buf_3km_null_or_ge081
Added field evt_buf_3_5km_null_or_ge081
Added field evt_buf_4km_null_or_ge081


In [50]:
import arcpy
from datetime import datetime

gdb = r"C:\Users\Melanie\Desktop\Rx_ClassiFIRE\ClassiFIRE_Rx_burn_permits.gdb"
events_fc = fr"{gdb}\SEFM_events_2010_2022_prescribed_FL"

buffer_labels = [
    "0_5km", "1km", "1_5km", "2km",
    "2_5km", "3km", "3_5km", "4km"
]

# Two permit subsets → two independent event-side match runs
permit_sets = {
    "ge081": "FL_ge081",
    "null_or_ge081": "FL_null_or_ge081"
}

# --- DATE PARSERS ---

def parse_sefm_date(val):
    s = str(val)
    return datetime(int(s[0:4]), int(s[4:6]), int(s[6:8]))

def parse_permit_date(val):
    if isinstance(val, datetime):
        return val
    return datetime.strptime(val, "%m/%d/%Y")

# --- MAIN LOOP FOR BOTH FLORIDA PERMIT DATASETS ---

for suffix, pset_label in permit_sets.items():

    print(f"\n=== Processing event matching for permit set: {pset_label} ===")

    for label in buffer_labels:

        print(f"  Buffer {label}...")

        # Spatial join table for this permit set + buffer
        sj = fr"{gdb}\PermitEventJoin_{label}_{pset_label}"

        # Build mapping: event OBJECTID → list of permit dates
        event_to_permits = {}

        with arcpy.da.SearchCursor(
            sj,
            ["JOIN_FID", "DATE"]
        ) as cur:
            for eid, permit_date_raw in cur:
                if permit_date_raw is None:
                    continue
                permit_date = parse_permit_date(permit_date_raw)
                event_to_permits.setdefault(eid, []).append(permit_date)

        # Field to update on the event layer
        field = f"evt_buf_{label}_{suffix}"

        with arcpy.da.UpdateCursor(
            events_fc,
            ["OBJECTID", "MIN_prebd_min_corrected", "MAX_bd_min_corrected_plus8", field]
        ) as cur:

            for eid, tmin_raw, tmax_raw, _ in cur:

                # If event has no temporal window, it cannot match
                if tmin_raw is None or tmax_raw is None:
                    cur.updateRow([eid, tmin_raw, tmax_raw, "no_match"])
                    continue

                tmin = parse_sefm_date(tmin_raw)
                tmax = parse_sefm_date(tmax_raw)

                classification = "no_match"

                # Only check temporal overlap if spatial matches exist
                if eid in event_to_permits:
                    for permit_date in event_to_permits[eid]:
                        if tmin <= permit_date <= tmax:
                            classification = "match"
                            break

                cur.updateRow([eid, tmin_raw, tmax_raw, classification])

        print(f"    Completed buffer {label}")

print("\nAll Florida event matching complete.")


=== Processing event matching for permit set: FL_ge081 ===
  Buffer 0_5km...
    Completed buffer 0_5km
  Buffer 1km...
    Completed buffer 1km
  Buffer 1_5km...
    Completed buffer 1_5km
  Buffer 2km...
    Completed buffer 2km
  Buffer 2_5km...
    Completed buffer 2_5km
  Buffer 3km...
    Completed buffer 3km
  Buffer 3_5km...
    Completed buffer 3_5km
  Buffer 4km...
    Completed buffer 4km

=== Processing event matching for permit set: FL_null_or_ge081 ===
  Buffer 0_5km...
    Completed buffer 0_5km
  Buffer 1km...
    Completed buffer 1km
  Buffer 1_5km...
    Completed buffer 1_5km
  Buffer 2km...
    Completed buffer 2km
  Buffer 2_5km...
    Completed buffer 2_5km
  Buffer 3km...
    Completed buffer 3km
  Buffer 3_5km...
    Completed buffer 3_5km
  Buffer 4km...
    Completed buffer 4km

All Florida event matching complete.


In [51]:
import arcpy

gdb = r"C:\Users\Melanie\Desktop\Rx_ClassiFIRE\ClassiFIRE_Rx_burn_permits.gdb"
events_fc = fr"{gdb}\SEFM_events_2010_2022_prescribed_FL"

buffer_labels = [
    "0_5km", "1km", "1_5km", "2km",
    "2_5km", "3km", "3_5km", "4km"
]

# Two permit subsets → two sets of event-side fields
suffixes = [
    "ge081",
    "null_or_ge081"
]

total = int(arcpy.management.GetCount(events_fc)[0])

for suf in suffixes:

    print(f"\n=== Event match percentages for FL_{suf} ===")

    results = {}

    for label in buffer_labels:
        field = f"evt_buf_{label}_{suf}"
        match_count = 0

        with arcpy.da.SearchCursor(events_fc, [field]) as cur:
            for (val,) in cur:
                if val == "match":
                    match_count += 1

        pct = (match_count / total) * 100
        results[label] = pct
        print(f"{label}: {pct:.2f}% matched")

    print("Summary:", results)


=== Event match percentages for FL_ge081 ===
0_5km: 22.10% matched
1km: 33.88% matched
1_5km: 40.80% matched
2km: 45.53% matched
2_5km: 49.16% matched
3km: 52.17% matched
3_5km: 54.83% matched
4km: 57.23% matched
Summary: {'0_5km': 22.104517454762117, '1km': 33.88020698941126, '1_5km': 40.80393711500298, '2km': 45.5306332130743, '2_5km': 49.161861778739016, '3km': 52.16724294045066, '3_5km': 54.82592152980751, '4km': 57.225159628620084}

=== Event match percentages for FL_null_or_ge081 ===
0_5km: 24.85% matched
1km: 38.95% matched
1_5km: 48.02% matched
2km: 54.62% matched
2_5km: 59.84% matched
3km: 64.14% matched
3_5km: 67.88% matched
4km: 71.04% matched
Summary: {'0_5km': 24.85233472983941, '1km': 38.95360565983739, '1_5km': 48.02323235640461, '2km': 54.616680193853185, '2_5km': 59.83879972413922, '3km': 64.13856843270733, '3_5km': 67.88473898785368, '4km': 71.04212506626759}
